This code bootstraps wind data by cluster

In [1]:
import numpy as np
import pandas as pd, os, datetime

# Statistical analysis
from typing import Sequence, Tuple, Optional
from joblib import Parallel, delayed
from scipy import stats
import math

import matplotlib.pyplot as plt

# Import the loader function
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
from process_code import load_generation_data

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that i

Dask dashboard: /proxy/8787/status


2025-12-11 10:48:06,006 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle cee544f46277e03a913ec0f30b38a86e initialized by task ('shuffle-transfer-cee544f46277e03a913ec0f30b38a86e', 0) executed on worker tcp://127.0.0.1:39207
2025-12-11 10:48:37,789 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle cee544f46277e03a913ec0f30b38a86e deactivated due to stimulus 'task-finished-1765410517.785932'
2025-12-11 10:48:56,252 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle d2bf34415c6f40f73dd94d7619c24bee initialized by task ('shuffle-transfer-d2bf34415c6f40f73dd94d7619c24bee', 1) executed on worker tcp://127.0.0.1:36239
2025-12-11 10:49:24,038 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle d2bf34415c6f40f73dd94d7619c24bee deactivated due to stimulus 'task-finished-1765410563.815298'


In [2]:
clust_map = pd.read_csv("data/preprocess/wind_spatial_clusters.csv")

In [ ]:
data, info = load_generation_data(
    sdate="2009-07-01",
    edate="2024-06-30",
    mode="hourly",
    ftype=["Wind"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=True,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

Read gen_details & hw_tseries with Dask: 0.07 sec
Select group: 0.00 sec
--- Starting Dask-Native Process ---


/g/data/ng72/ms5578/ID_HW_BARRA/process_code.py:168: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grp_pd_jittered = grp_pd.groupby(['lat', 'lon'], group_keys=False).apply(


Starting final Dask compute...


2025-12-11 10:50:00,430 - distributed.worker - ERROR - failed during get data with tcp://127.0.0.1:33697 -> tcp://127.0.0.1:39207
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/tornado/iostream.py", line 861, in _read_to_buffer
    bytes_read = self.read_from_fd(buf)
                 ^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/tornado/iostream.py", line 1113, in read_from_fd
    return self.socket.recv_into(buf, len(buf))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TimeoutError: [Errno 110] Connection timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/worker.py", line 1795, in get_data
    response = await comm.read(deserializers=serializers)
               ^^^^^^^

Append clusters to generation timeseries in tidy format

In [ ]:
df = pd.merge(data,clust_map[['DUID','cluster']], on='DUID').sort_values(['EHF_flag','cluster','DUID','time'])

Z-score normalise by cluster.

In [ ]:
df['z_score'] = df.groupby('DUID')['TOTALMWh'].transform(
    lambda x: (x - x.mean()) / x.std(ddof=1)
)
df

In [ ]:
def bootstrap_curve(
    X: np.ndarray,
    n_boot: int,
    batch_size: Optional[int] = None,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Bootstrap mean and median daily curves with confidence intervals, batched for speed and memory.

    Parameters
    ----------
    X : np.ndarray
        Array of shape (n_days, n_hours)
    n_boot : int
        Number of bootstrap samples
    batch_size : Optional[int]
        Number of bootstrap samples to process per batch. If None, chosen automatically.
    seed : int
        RNG seed for reproducibility

    Returns
    -------
    mean_curve : np.ndarray, shape (n_hours,)
    mean_lower : np.ndarray, shape (n_hours,)
    mean_upper : np.ndarray, shape (n_hours,)
    median_curve : np.ndarray, shape (n_hours,)
    median_lower : np.ndarray, shape (n_hours,)
    median_upper : np.ndarray, shape (n_hours,)
    """
    n_days, n_hours = X.shape
    curves_mean = np.empty((n_boot, n_hours), dtype=np.float64)
    curves_median = np.empty((n_boot, n_hours), dtype=np.float64)
    rng = np.random.default_rng(seed)

    # Choose a batch size that keeps memory reasonable
    if batch_size is None:
        # Aim for ~50–200 MB peak for the 3D array; adjust as needed
        # Each float64 ~8 bytes -> batch_size * n_days * n_hours * 8 bytes
        target_bytes = 100 * 1024 * 1024  # 100 MB
        approx_bs = max(1, int(target_bytes / (n_days * n_hours * 8)))
        batch_size = min(n_boot, max(100, approx_bs))

    out_start = 0
    while out_start < n_boot:
        k = min(batch_size, n_boot - out_start)
        # Indices: shape (k, n_days)
        idx = rng.integers(0, n_days, size=(k, n_days))
        # Samples: shape (k, n_days, n_hours)
        samples = X[idx]
        # Aggregate across days -> shape (k, n_hours)
        curves_mean[out_start:out_start + k] = samples.mean(axis=1)
        curves_median[out_start:out_start + k] = np.median(samples, axis=1)
        out_start += k

    # Compute mean and 95% CI across bootstrap replicates
    mean_curve = curves_mean.mean(axis=0)
    mean_lower = np.percentile(curves_mean, 2.5, axis=0)
    mean_upper = np.percentile(curves_mean, 97.5, axis=0)

    # Median curve is the median across bootstrap medians
    median_curve = np.median(curves_median, axis=0)
    median_lower = np.percentile(curves_median, 2.5, axis=0)
    median_upper = np.percentile(curves_median, 97.5, axis=0)

    return mean_curve, mean_lower, mean_upper, median_curve, median_lower, median_upper

In [ ]:
def process_group(name, gdf: pd.DataFrame, values: str = 'z_score', n_boot: int = 1000, batch_size: Optional[int] = None):
    """
    Compute bootstrapped curves for a single grouped DataFrame.
    Returns:
        name (group key), curves tuple, hour_labels (array)
    """
    gdf = gdf.copy()
    gdf['day'] = gdf['time'].dt.floor('D')
    gdf['hour'] = gdf['time'].dt.hour

    daily_matrix = (
        gdf.pivot_table(index='day', columns='hour', values=values)
           .dropna()
           .sort_index()
    )

    if daily_matrix.empty:
        hour_labels = np.array(sorted(gdf['hour'].unique()))
        n_hours = len(hour_labels)
        empty = np.full(n_hours, np.nan)
        return name, (empty, empty, empty, empty, empty, empty), hour_labels

    X = daily_matrix.to_numpy()
    curves = bootstrap_curve(X, n_boot=n_boot, batch_size=batch_size, seed=42)
    hour_labels = daily_matrix.columns.to_numpy()
    return name, curves, hour_labels

In [ ]:
def compute_bootstrap_df_joblib(
    df: pd.DataFrame,
    values: str = 'z_score',
    n_boot: int = 1000,
    group_cols: Sequence[str] = ('cluster', 'DUID', 'EHF_flag'),
    n_jobs: int = -1,  # use all cores by default
    backend: str = 'loky',  # 'loky' for processes (robust in notebooks), or 'threading'
    prefer: Optional[str] = None,  # 'processes' or 'threads'
    batch_size: Optional[int] = None,
) -> pd.DataFrame:
    """
    Notebook-friendly parallel computation of bootstrapped curves using joblib.

    Parameters
    ----------
    df : pd.DataFrame
        Must include columns: 'time' (datetime-like), group_cols, and `values`.
    values : str
        Column name to aggregate (default 'z_score').
    n_boot : int
        Number of bootstrap samples.
    group_cols : Sequence[str]
        Grouping columns, e.g. ('cluster','DUID','EHF_flag') or ('cluster','EHF_flag').
    n_jobs : int
        Number of parallel workers. -1 means all available cores.
    backend : str
        'loky' for separate processes (better isolation), or 'threading'.
    prefer : Optional[str]
        Hint for joblib scheduler: 'processes' or 'threads'. Usually not needed.
    batch_size : Optional[int]
        Bootstrap batch size; see `bootstrap_curve`.

    Returns
    -------
    pd.DataFrame with columns:
        * group_cols...
        * hour
        * mean, mean_lower, mean_upper
        * median, median_lower, median_upper
    """
    needed = set(group_cols) | {'time', values}
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    grouped = df.groupby(list(group_cols), sort=False)

    # Build tasks
    tasks = ((name, gdf, values, n_boot, batch_size) for name, gdf in grouped)

    # Run in parallel
    results = Parallel(n_jobs=n_jobs, backend=backend, prefer=prefer)(
        delayed(process_group)(name, gdf, values, n_boot, batch_size)
        for (name, gdf, values, n_boot, batch_size) in tasks
    )

    # Assemble rows
    rows = []
    for name, curves, hour_labels in results:
        mean_curve, mean_lower, mean_upper, median_curve, median_lower, median_upper = curves

        if isinstance(name, tuple):
            key_map = dict(zip(group_cols, name))
        else:
            key_map = {group_cols[0]: name}

        for i, hour in enumerate(hour_labels):
            rows.append({
                **key_map,
                'hour': int(hour),
                'mean': float(mean_curve[i]) if np.isfinite(mean_curve[i]) else np.nan,
                'mean_lower': float(mean_lower[i]) if np.isfinite(mean_lower[i]) else np.nan,
                'mean_upper': float(mean_upper[i]) if np.isfinite(mean_upper[i]) else np.nan,
                'median': float(median_curve[i]) if np.isfinite(median_curve[i]) else np.nan,
                'median_lower': float(median_lower[i]) if np.isfinite(median_lower[i]) else np.nan,
                'median_upper': float(median_upper[i]) if np.isfinite(median_upper[i]) else np.nan,
            })

    return pd.DataFrame(rows)

Here is where the grouping, value, and number of bootstraps is specified.

In [ ]:
def bootstrap_diff(df, n_boot=1000, values='z_score'):
    hw = df[df["EHF_flag"] == 1][values].values
    bl = df[df["EHF_flag"] == 0][values].values

    bootstrap_differences = []
    
    for i in range(n_boot):
        # Resample with replacement from each dataset
        sample_hw = np.random.choice(hw, size=len(hw), replace=True)
        sample_bl = np.random.choice(bl, size=len(bl), replace=True)

        # Calculate the difference in means for the current bootstrap sample
        bootstrap_differences.append(sample_hw.mean() - sample_bl.mean())

    return bootstrap_differences

In [ ]:
x = df.groupby('DUID')[['EHF_flag','z_score']].apply(bootstrap_diff, include_groups=True)

In [ ]:
x

In [ ]:
df_results = compute_bootstrap_df_joblib(
    df,
    values='z_score',
    n_boot=1000,
    group_cols=('cluster', 'DUID', 'EHF_flag'),
    n_jobs=-1,
    backend='loky',
)

In [ ]:
df_results

In [ ]:
def plot_all_clusters_faceted(df_results: pd.DataFrame, metric='mean'):
    """
    Plot mean/median curves with 95% CI for all clusters in a 2x3 faceted layout.
    Each facet shows all DUIDs within the cluster, split by EHF_flag.

    Parameters
    ----------
    df_results : pd.DataFrame
        Columns needed:
        ['cluster','DUID','EHF_flag','hour',
         'mean','mean_lower','mean_upper',
         'median','median_lower','median_upper']
    metric : str
        'mean' or 'median'
    """
    if metric not in ['mean', 'median']:
        raise ValueError("metric must be 'mean' or 'median'")

    # Determine up to 6 clusters (or all if <= 6)
    clusters = sorted(df_results['cluster'].unique())
    n = len(clusters)
    if n == 0:
        raise ValueError("No clusters found in df_results")

    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5*ncols, 3.8*nrows), sharex=True, sharey=True)
    axes = np.array(axes).reshape(-1)

    label_map = {0.0: "No heatwave", 1.0: "Heatwave"}
    label_map_int = {0: "No heatwave", 1: "Heatwave"}
    colors = ['blue', 'orange'] if metric == "mean" else ['green', 'red']

    # Plot each cluster in its own facet
    for ax_idx, cluster_name in enumerate(clusters[:nrows*ncols]):
        ax = axes[ax_idx]
        cluster_df = df_results[df_results['cluster'] == cluster_name].copy()
        duids = sorted(cluster_df['DUID'].unique())

        # Plot each DUID's curves by EHF_flag
        for duid in duids:
            sub = cluster_df[cluster_df['DUID'] == duid]
            hw_statuses = sorted(sub['EHF_flag'].unique())

            for i, status in enumerate(hw_statuses):
                status_df = sub[sub['EHF_flag'] == status].sort_values('hour')

                y = status_df[metric].values
                lower = status_df[f'{metric}_lower'].values
                upper = status_df[f'{metric}_upper'].values

                label = label_map.get(float(status), label_map_int.get(int(status), str(status)))
                ax.plot(status_df['hour'], y, color=colors[i % len(colors)], alpha=0.9, label=label)
                ax.fill_between(status_df['hour'], lower, upper, color=colors[i % len(colors)], alpha=0.18)

        ax.set_title(f"Cluster {cluster_name}")
        ax.set_xticks(range(0, 24))
        ax.grid(True, alpha=0.2)

    # Remove unused axes if clusters < 6
    for ax in axes[len(clusters):]:
        ax.remove()

    fig.suptitle(f'{metric.capitalize()} daily generation profiles with CI — All Clusters', y=0.98)
    fig.text(0.5, 0.04, 'Hour of Day', ha='center')
    fig.text(0.02, 0.5, 'Value', va='center', rotation='vertical')

    # Shared legend for EHF statuses (avoid duplicates)
    handles, labels = axes[0].get_legend_handles_labels()
    # Deduplicate labels while preserving order
    seen = set()
    uniq = [(h, l) for h, l in zip(handles, labels) if not (l in seen or seen.add(l))]
    if uniq:
        fig.legend([h for h, _ in uniq], [l for _, l in uniq], title="EHF status",
                   loc='upper right', bbox_to_anchor=(0.98, 0.98))
    plt.tight_layout(rect=[0.03, 0.05, 0.98, 0.93])
    plt.show()

# Example:
plot_all_clusters_faceted(df_results, metric='mean')
plot_all_clusters_faceted(df_results, metric='median')


In [ ]:
def plot_cluster_duids_with_ci(df_results: pd.DataFrame, cluster_name, metric='mean'):
    """
    Plot mean/median curves with 95% CI for every DUID within a cluster,
    each DUID in its own subplot, split by EHF_flag.

    Parameters
    ----------
    df_results : pd.DataFrame
        Columns needed:
        ['cluster','DUID','EHF_flag','hour',
         'mean','mean_lower','mean_upper',
         'median','median_lower','median_upper']
    cluster_name : hashable
        Cluster identifier to filter.
    metric : str
        'mean' or 'median'
    """
    if metric not in ['mean', 'median']:
        raise ValueError("metric must be 'mean' or 'median'")

    cluster_df = df_results[df_results['cluster'] == cluster_name].copy()
    if cluster_df.empty:
        raise ValueError(f"No data for cluster {cluster_name}")

    duids = sorted(cluster_df['DUID'].unique())
    n = len(duids)
    ncols = 3
    nrows = math.ceil(n / ncols)

    label_map = {0.0: "No heatwave", 1.0: "Heatwave"}
    label_map_int = {0: "No heatwave", 1: "Heatwave"}
    colors = ['blue', 'orange'] if metric == "mean" else ['green', 'red']

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.5*ncols, 3.2*nrows), sharex=True, sharey=True)
    axes = np.array(axes).reshape(-1)

    for ax, duid in zip(axes, duids):
        sub = cluster_df[cluster_df['DUID'] == duid]
        hw_statuses = sorted(sub['EHF_flag'].unique())

        for i, status in enumerate(hw_statuses):
            status_df = sub[sub['EHF_flag'] == status].sort_values('hour')

            y = status_df[metric].values
            lower = status_df[f'{metric}_lower'].values
            upper = status_df[f'{metric}_upper'].values

            label = label_map.get(float(status), label_map_int.get(int(status), str(status)))
            ax.plot(status_df['hour'], y, color=colors[i % len(colors)], label=label)
            ax.fill_between(status_df['hour'], lower, upper, color=colors[i % len(colors)], alpha=0.25)

        ax.set_title(f"DUID {duid}")
        ax.set_xticks(range(0, 24))

    # Remove unused axes if any
    for ax in axes[len(duids):]:
        ax.remove()

    fig.suptitle(f'{metric.capitalize()} daily generation profiles with CI — Cluster {cluster_name}', y=0.98)
    fig.text(0.5, 0.04, 'Hour of Day', ha='center')
    fig.text(0.02, 0.5, 'Value', va='center', rotation='vertical')

    # One shared legend for EHF statuses
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="EHF status", loc='upper right', bbox_to_anchor=(0.98, 0.98))
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Example:
plot_cluster_duids_with_ci(df_results, cluster_name=5, metric='mean')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.transforms import IdentityTransform


def _bounding_box_pixels(ab, renderer):
    """
    Compute bounding box of an AnnotationBbox *in figure pixel coordinates*.
    """
    # Get full bbox of the annotation box (including offset)
    bb = ab.get_window_extent(renderer=renderer)
    return bb.x0, bb.y0, bb.x1, bb.y1


def _intersects(bb1, bb2):
    """Return True if two bounding boxes intersect."""
    x0a, y0a, x1a, y1a = bb1
    x0b, y0b, x1b, y1b = bb2
    return not (x1a < x0b or x1b < x0a or y1a < y0b or y1b < y0a)


def plot_cluster_duid_curves_on_world_map_far_boxes_collision(
    df_results: pd.DataFrame,
    df_coords: pd.DataFrame,
    cluster_name,
    metric='mean',
    status_order=(0.0, 1.0),
    colors=('blue', 'orange'),
    image_zoom=0.75,
    per_duid_ylim=False,
    annotate_duid=True,
    extent=None,

    # placement parameters
    base_radius=200,           # initial offset in points
    radius_step=40,            # grow radius when colliding
    angle_step_deg=30,         # step angle when colliding
    max_radius_attempts=50,    # safety cap to avoid infinite loop
):
    """
    WORLD MAP + many small inset charts.
    Collision-free placement in screen space.

    Parameters identical to your original function except:
        - No need for angles_deg list.
        - Collision detection automatically manages position.
    """

    if metric not in ['mean', 'median']:
        raise ValueError("metric must be 'mean' or 'median'")

    # --- Filter cluster ---
    dfc = df_results[df_results['cluster'] == cluster_name].copy()
    if dfc.empty:
        raise ValueError(f"No df_results data for cluster {cluster_name}")

    coords_c = df_coords.copy()
    if 'cluster' in coords_c.columns:
        coords_c = coords_c[coords_c['cluster'] == cluster_name].copy()

    duids_present = dfc['DUID'].unique()
    coords_c = coords_c[coords_c['DUID'].isin(duids_present)].dropna(subset=['lat', 'lon'])
    if coords_c.empty:
        raise ValueError(f"No coordinates for cluster {cluster_name} DUIDs")

    # --- Shared ylim ---
    shared_ylim = None
    if not per_duid_ylim:
        ys = []
        for duid in duids_present:
            sub = dfc[dfc['DUID'] == duid]
            for status in status_order:
                ssub = sub[sub['EHF_flag'] == status].sort_values('hour')
                if not ssub.empty:
                    ys.append(ssub[metric].values)
                    ys.append(ssub[f'{metric}_lower'].values)
                    ys.append(ssub[f'{metric}_upper'].values)

        if ys:
            ycat = np.concatenate([y for y in ys if y.size])
            ymin, ymax = np.nanmin(ycat), np.nanmax(ycat)
            pad = 0.05 * (ymax - ymin if ymax > ymin else 1)
            shared_ylim = (ymin - pad, ymax + pad)

    # --- Map setup ---
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(12, 7))
    ax = plt.axes(projection=proj)
    ax.add_feature(cfeature.LAND, facecolor="#f0efe7", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="#dbeaf6", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=1)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, zorder=1)

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                      alpha=0.5, linestyle='--')
    gl.right_labels = False
    gl.top_labels = False

    # --- Extent ---
    if extent is None:
        min_lat, max_lat = coords_c['lat'].min(), coords_c['lat'].max()
        min_lon, max_lon = coords_c['lon'].min(), coords_c['lon'].max()
        ax.set_extent([
            min_lon - 1.0, max_lon + 1.0,
            min_lat - 1.0, max_lat + 1.0,
        ], crs=proj)
    else:
        ax.set_extent(extent, crs=proj)

    # Scatter actual locations
    ax.scatter(coords_c['lon'], coords_c['lat'], s=16, color='k',
               alpha=0.65, transform=proj, zorder=3)

    # For collision checking
    renderer = fig.canvas.get_renderer()
    placed_boxes = []  # list of bounding boxes in pixel space

    # Process each DUID
    for _, row in coords_c.iterrows():
        duid = row['DUID']
        lon = row['lon']
        lat = row['lat']

        sub = dfc[dfc['DUID'] == duid]
        if sub.empty:
            continue

        # --- Build mini-plot image ---
        mini_fig, mini_ax = plt.subplots(figsize=(2.4, 1.4), dpi=100)
        mini_canvas = FigureCanvas(mini_fig)

        mini_ax.set_facecolor('white')
        mini_ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)

        for i, status in enumerate(status_order):
            ssub = sub[sub['EHF_flag'] == status].sort_values('hour')
            if ssub.empty:
                continue
            hrs = ssub['hour'].values
            y = ssub[metric].values
            lo = ssub[f'{metric}_lower'].values
            hi = ssub[f'{metric}_upper'].values
            color = colors[i % len(colors)]

            mini_ax.plot(hrs, y, color=color, lw=1.6)
            mini_ax.fill_between(hrs, lo, hi, color=color, alpha=0.22)

        mini_ax.set_xticks([0, 6, 12, 18, 23])
        mini_ax.set_xlim(0, 23)

        if shared_ylim:
            mini_ax.set_ylim(*shared_ylim)
        else:
            lines = mini_ax.get_lines()
            if lines:
                ydata = np.concatenate([ln.get_ydata() for ln in lines])
                ymin, ymax = np.nanmin(ydata), np.nanmax(ydata)
                pad = 0.05 * (ymax - ymin if ymax > ymin else 1)
                mini_ax.set_ylim(ymin - pad, ymax + pad)

        for s in ["top", "right"]:
            mini_ax.spines[s].set_visible(False)

        mini_ax.tick_params(labelsize=6)
        mini_ax.set_xlabel('')
        mini_ax.set_ylabel('')
        mini_canvas.draw()
        img = np.asarray(mini_canvas.buffer_rgba())
        plt.close(mini_fig)

        oi = OffsetImage(img, zoom=image_zoom)

        # === COLLISION-FREE PLACEMENT ======================================

        angle = 0.0  # radians
        radius = base_radius
        placed = False

        for attempt in range(max_radius_attempts):

            dx = radius * np.cos(angle)
            dy = radius * np.sin(angle)

            ab = AnnotationBbox(
                oi,
                (lon, lat),
                xycoords='data',
                xybox=(dx, dy),
                boxcoords="offset points",
                arrowprops=dict(arrowstyle="-", color='gray',
                                lw=1.0, shrinkA=0, shrinkB=5),
                bboxprops=dict(edgecolor='gray', facecolor='white',
                               alpha=0.92),
                pad=0.25,
                zorder=4,
                clip_on=False
            )

            # temporarily draw to get bbox
            ax.add_artist(ab)
            fig.canvas.draw_idle()
            bb = _bounding_box_pixels(ab, renderer)

            # collision test
            if not any(_intersects(bb, old) for old in placed_boxes):
                placed_boxes.append(bb)
                placed = True
                break

            # collision → remove artist and try again
            ab.remove()
            angle += np.deg2rad(angle_step_deg)
            radius += radius_step

        if not placed:
            print(f"Warning: Could not place box for {duid} without collision.")

        # --- DUID label near box ---
        if annotate_duid and placed:
            ax.annotate(
                duid,
                xy=(lon, lat),
                xycoords='data',
                xytext=(dx + 6, dy - 2),
                textcoords='offset points',
                fontsize=8,
                color='black',
                zorder=5,
                clip_on=False
            )

    # Status legend
    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], color=colors[i], lw=2)
               for i, _ in enumerate(status_order)]
    labels = ["No heatwave", "Heatwave"]
    ax.legend(handles, labels, title="EHF status", loc='lower left')

    plt.title(f"{metric.capitalize()} daily profiles with CI — Cluster {cluster_name}")
    plt.subplots_adjust(left=0.05, right=0.98, top=0.92, bottom=0.06)
    plt.show()


In [ ]:
plot_cluster_duid_curves_on_world_map_far_boxes_collision(
    df_results=df_results,
    df_coords=clust_map,
    cluster_name=3,
    metric="mean",
    status_order=(0.0, 1.0),
    colors=("blue", "orange"),
    image_zoom=0.75,
    per_duid_ylim=False,
    annotate_duid=True,

    base_radius=200,
    radius_step=40,
    angle_step_deg=30,
    max_radius_attempts=50
)